# Imports

In [ ]:
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
from jax import vmap
import jax.tree_util as jtu
from src.experiment import (
    ElectronReactionSamplingExperiment,
    HeterogeneousReactionSamplingExperiment,
    EqualDiffusionReactionSamplingExperiment,
)
from src.fdm import (
    AdsorptionReactionBackwardImplicitFDSolver,
    AdsorptionReactionExplicitFDSolver,
    AdsorptionReactionNewtonFDSolver,
    ElectronReactionFDSolver,
    HeterogeneousReactionFDSolver,
    EqualDiffusionReactionFDSolver,
)
from src.params import (
    AdsorptionReactionParams,
    ElectronReactionParams,
    HeterogenousReactionParams,
    EqualDiffusionReactionParams,
)
from src.utils import generate_noisy_samples
from src.voltammetry import CyclicDC
import blackjax

sns.set_theme()
sns.set_context("paper", font_scale=1.5)

# Equal Diffusion Reaction

In [ ]:
hmc = np.load("./data/D_HMC_0.10_100.npz")
rw = np.load("./data/D_RW_0.10_100.npz")
true_params = EqualDiffusionReactionSamplingExperiment().true_parameters

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(10, 3))

options = {"density": True, "bins": 50, "alpha": 0.8, "histtype": "step"}

ax0.set_title(r"$\alpha$")
ax0.set_ylabel("Density")
ax0.hist(hmc["alpha"].flatten(), **options, label="HMC")
ax0.hist(rw["alpha"].flatten(), **options, label="RW")
ax0.axvline(x=true_params.alpha, linestyle="--", color="black", label="True Value")

ax1.set_title(r"$K_0$")
ax1.hist(hmc["K0"].flatten(), **options)
ax1.hist(rw["K0"].flatten(), **options)
ax1.axvline(x=true_params.K0, linestyle="--", color="black")

ax2.set_title(r"$E_f$")
ax2.hist(hmc["Ef"].flatten(), **options)
ax2.hist(rw["Ef"].flatten(), **options)
ax2.axvline(x=true_params.Ef, linestyle="--", color="black")


handles, labels = ax0.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)

plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()

## Biplot

In [ ]:
hmc = np.load("./data/D_HMC_0.10_100.npz")
df_hmc = pd.DataFrame({k: hmc[k].flatten() for k in hmc.files})

label_map = {
    "alpha": r"$\alpha$",
    "K0": r"$K_0$",
    "Ef": r"$E_f$",
}

g = sns.PairGrid(
    df_hmc.drop(columns=["logdensity"]).rename(columns=label_map),
    corner=True,
    layout_pad=True,
    diag_sharey=False,
)

g.map_diag(sns.kdeplot, common_norm=False, fill=True, alpha=0.5)
g.map_lower(sns.kdeplot)

plt.tight_layout()
plt.show()

# Electron Transfer

$
\mathrm{A}+\mathrm{e}^{-} \rightleftharpoons \mathrm{B}
$

## Example Voltammagram

In [ ]:
voltammetry = CyclicDC()

fdm_solver = ElectronReactionFDSolver(voltammetry, dtheta=2.5e-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

# Irreversible

irre_params = ElectronReactionParams(
    alpha=jnp.array(0.7),
    K0=jnp.array(1.0),
    Ef=jnp.array(0.0),
    dB=jnp.array(1.0),
)

irre_current = fdm_solver.solve(irre_params)

ax1.plot(fdm_solver.applied_potentials, irre_current)
ax1.axhline(
    y=-0.496 * jnp.sqrt(irre_params.alpha) * jnp.sqrt(voltammetry.sigma),
    linestyle="--",
    c="red",
)
ax1.axvline(
    x=(jnp.log(irre_params.K0 / jnp.sqrt(irre_params.alpha * voltammetry.sigma)) - 0.78)
    / irre_params.alpha,
    linestyle="--",
    c="red",
)

rev_params = ElectronReactionParams(
    alpha=jnp.array(0.7),
    K0=jnp.array(100000.0),
    Ef=jnp.array(0.0),
    dB=jnp.array(1.0),
)

rev_current = fdm_solver.solve(rev_params)

ax2.plot(fdm_solver.applied_potentials, rev_current)
ax2.axhline(y=-0.446 * jnp.sqrt(voltammetry.sigma), linestyle="--", c="red")
ax2.set_title("Reversible")
ax2.set_xlabel(r"$\theta$")

ax1.set_xlabel(r"$\theta$")
ax1.set_ylabel(r"$J$")
ax1.set_title("Irreversible")

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Effect of each parameter

In [ ]:
base_params = ElectronReactionParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(5.0),
    Ef=jnp.array(0.5),
    dB=jnp.array(0.8),
)


def electron_transfer_coef(axs, sigma):
    ax1, ax2, ax3 = axs

    voltammetry = CyclicDC(sigma=sigma)
    fd_solver = ElectronReactionFDSolver(voltammetry)

    # alpha
    alpha_range = jnp.array([0.3, 0.4, 0.6, 0.7])
    alpha_params = ElectronReactionParams(
        alpha=alpha_range,
        K0=jnp.full_like(alpha_range, base_params.K0),
        Ef=jnp.full_like(alpha_range, base_params.Ef),
        dB=jnp.full_like(alpha_range, base_params.dB),
    )

    alpha_currents = vmap(fd_solver.solve)(alpha_params)
    for val, current in zip(alpha_range, alpha_currents):
        ax1.plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

    # K0
    K0_range = jnp.array([1.0, 5.0, 20.0, 50.0])

    K0_params = ElectronReactionParams(
        alpha=jnp.full_like(K0_range, base_params.alpha),
        K0=K0_range,
        Ef=jnp.full_like(K0_range, base_params.Ef),
        dB=jnp.full_like(K0_range, base_params.dB),
    )

    K0_currents = vmap(fd_solver.solve)(K0_params)
    for val, current in zip(K0_range, K0_currents):
        ax2.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

    # dB
    dB_range = jnp.array([0.1, 0.3, 0.6, 0.8])

    dB_params = ElectronReactionParams(
        alpha=jnp.full_like(dB_range, base_params.alpha),
        K0=jnp.full_like(dB_range, base_params.K0),
        Ef=jnp.full_like(dB_range, base_params.Ef),
        dB=dB_range,
    )

    dB_currents = vmap(fd_solver.solve)(dB_params)
    for val, current in zip(dB_range, dB_currents):
        ax3.plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")


fig, axs = plt.subplots(2, 3, figsize=(12, 8), sharex=True, sharey="row")
row_1 = axs[0, 0], axs[0, 1], axs[0, 2]
row_2 = axs[1, 0], axs[1, 1], axs[1, 2]

electron_transfer_coef(row_1, 10)
electron_transfer_coef(row_2, 1000)

axs[0, 0].legend()
axs[0, 0].set_title(r"$\alpha$")
axs[0, 0].set_ylabel(r"$J$")
axs[1, 0].set_ylabel(r"$J$")
axs[1, 0].set_xlabel(r"$\theta$")
axs[0, 0].yaxis.set_inverted(True)
axs[1, 0].yaxis.set_inverted(True)

axs[0, 1].legend()
axs[0, 1].set_title(r"$K_0$")
axs[1, 1].set_xlabel(r"$\theta$")
axs[0, 1].yaxis.set_inverted(True)
axs[1, 1].yaxis.set_inverted(True)

axs[0, 2].legend()
axs[0, 2].set_title(r"$d_B$")
axs[1, 2].set_xlabel(r"$\theta$")
axs[0, 2].yaxis.set_inverted(True)
axs[1, 2].yaxis.set_inverted(True)

plt.gca().invert_xaxis()
plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

## Noisy Sample Data

In [ ]:
params = ElectronReactionParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(5.0),
    Ef=jnp.array(0.5),
    dB=jnp.array(0.8),
)


def plot_noise(sigma, ax, key):
    voltammetry = CyclicDC(sigma=sigma)
    fd_solver = ElectronReactionFDSolver(voltammetry)
    base_current = fd_solver.solve(params)

    for sigma in [0.2, 0.1, 0.05, 0.01]:
        noise_key, key = jr.split(key)
        noisy_current = generate_noisy_samples(
            1, base_current, sigma=sigma, key=noise_key
        )[0]
        ax.plot(fd_solver.applied_potentials, noisy_current, label=sigma)

    ax.yaxis.set_inverted(True)


key = jr.key(0)
key_1, key_2 = jr.split(key)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharex=True)

plot_noise(10, ax1, key_1)
plot_noise(1000, ax2, key_2)

ax1.set_ylabel(r"$J$")
ax1.set_xlabel(r"$\theta$")
ax2.set_xlabel(r"$\theta$")

plt.gca().invert_xaxis()

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## Histogram Comparison

In [ ]:
hmc = np.load("./data/E_HMC_0.10_10.npz")
rw = np.load("./data/E_RW_0.10_10.npz")
true_params = ElectronReactionSamplingExperiment().true_parameters

rw_burn_in = 0

fig, axs = plt.subplots(2, 2, figsize=(10, 6))

options = {"density": True, "bins": 50, "alpha": 0.8}

axs[0, 0].hist(hmc["alpha"].flatten(), **options, label="HMC")
axs[0, 0].hist(rw["alpha"].flatten(), **options, label="RW")
axs[0, 0].axvline(
    x=true_params.alpha, linestyle="--", color="black", label="True Value"
)

axs[0, 1].hist(hmc["K0"].flatten(), **options)
axs[0, 1].hist(rw["K0"].flatten(), **options)
axs[0, 1].set_xlim(0, 100)
axs[0, 1].axvline(x=true_params.K0, linestyle="--", color="black", label="True Value")

axs[1, 0].hist(hmc["Ef"].flatten(), **options)
axs[1, 0].hist(rw["Ef"].flatten(), **options)
axs[1, 0].axvline(x=true_params.Ef, linestyle="--", color="black", label="True Value")

axs[1, 1].hist(hmc["dB"].flatten(), **options)
axs[1, 1].hist(rw["dB"].flatten(), **options)
axs[1, 1].axvline(x=true_params.dB, linestyle="--", color="black", label="True Value")


handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
hmc = np.load("./data/E_HMC_0.10_1000.npz")
rw = np.load("./data/E_RW_0.10_1000.npz")

true_params = ElectronReactionSamplingExperiment().true_parameters

rw_burn_in = 160_000

fig, axs = plt.subplots(2, 2, figsize=(10, 6))

options = {"density": True, "bins": 50, "alpha": 0.8}

axs[0, 0].set_title(r"$\alpha$")
axs[0, 0].hist(hmc["alpha"].flatten(), **options, label="HMC")
axs[0, 0].hist(rw["alpha"].flatten()[rw_burn_in:], **options, label="RW")
axs[0, 0].axvline(
    x=true_params.alpha, linestyle="--", color="black", label="True Value"
)

axs[0, 1].set_title(r"$K_0$")
axs[0, 1].hist(hmc["K0"].flatten(), **options)
axs[0, 1].hist(rw["K0"].flatten()[rw_burn_in:], **options)
axs[0, 1].axvline(x=true_params.K0, linestyle="--", color="black", label="True Value")

axs[1, 0].set_title(r"$E_f$")
axs[1, 0].hist(hmc["Ef"].flatten(), **options)
axs[1, 0].hist(rw["Ef"].flatten()[rw_burn_in:], **options)
axs[1, 0].axvline(x=true_params.Ef, linestyle="--", color="black", label="True Value")

axs[1, 1].set_title(r"$d_B$")
axs[1, 1].hist(hmc["dB"].flatten(), **options)
axs[1, 1].hist(rw["dB"].flatten()[rw_burn_in:], **options)
axs[1, 1].axvline(x=true_params.dB, linestyle="--", color="black", label="True Value")


handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
rw = np.load("./data/E_RW_0.10_1000.npz")

true_params = ElectronReactionSamplingExperiment().true_parameters

fig, axs = plt.subplots(2, 2, figsize=(10, 6))

options = {"density": True, "bins": 50, "alpha": 0.8}

axs[0, 0].set_title(r"$\alpha$")
for i, chain in enumerate(hmc["alpha"]):
    axs[0, 0].hist(chain, **options, label=i + 1)
axs[0, 0].axvline(
    x=true_params.alpha, linestyle="--", color="black", label="True Value"
)

axs[0, 1].set_title(r"$K_0$")
for i, chain in enumerate(hmc["K0"]):
    axs[0, 1].hist(chain, **options, label=i + 1)
axs[0, 1].axvline(x=true_params.K0, linestyle="--", color="black", label="True Value")

axs[1, 0].set_title(r"$E_f$")
for i, chain in enumerate(hmc["Ef"]):
    axs[1, 0].hist(chain, **options, label=i + 1)
axs[1, 0].axvline(x=true_params.Ef, linestyle="--", color="black", label="True Value")

axs[1, 1].set_title(r"$d_B$")
for i, chain in enumerate(hmc["dB"]):
    axs[1, 1].hist(chain, **options, label=i + 1)
axs[1, 1].axvline(x=true_params.dB, linestyle="--", color="black", label="True Value")


handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=9)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Biplot

In [ ]:
hmc = np.load("./data/E_HMC_0.10_1000.npz")
df_hmc = pd.DataFrame({k: hmc[k].flatten() for k in hmc.files})

label_map = {
    "alpha": r"$\alpha$",
    "K0": r"$K_0$",
    "Ef": r"$E_f$",
    "dB": r"$d_B$",
}

g = sns.PairGrid(
    df_hmc.drop(columns=["logdensity"]).rename(columns=label_map),
    corner=True,
    layout_pad=True,
    diag_sharey=False,
)

g.map_diag(sns.kdeplot, common_norm=False, fill=True, alpha=0.5)
g.map_lower(sns.kdeplot)

experiment = ElectronReactionSamplingExperiment()

g.diag_axes[0].axvline(experiment.true_parameters.alpha, color="black", linestyle="--")
g.diag_axes[1].axvline(experiment.true_parameters.K0, color="black", linestyle="--")
g.diag_axes[2].axvline(experiment.true_parameters.Ef, color="black", linestyle="--")
g.diag_axes[3].axvline(experiment.true_parameters.dB, color="black", linestyle="--")

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


## Figure 6: Effect of noise

# Heterogeneous Reaction
$
\begin{gathered}
\mathrm{A}+\mathrm{e}^{-} \rightleftarrows \mathrm{B} \\
\mathrm{~B} \xrightarrow{k_{\text {het }}} \mathrm{C} \\
\mathrm{C}+\mathrm{e}^{-} \rightleftarrows \mathrm{D}
\end{gathered}
$

## Figure 7: Heterogeneous Parameter Effects

In [ ]:
voltammetry = CyclicDC()
fd_solver = HeterogeneousReactionFDSolver(voltammetry)

base_params = HeterogenousReactionParams(
    alpha_1=jnp.array(0.6),
    K0_1=jnp.array(10.0),
    Ef_1=jnp.array(0.5),
    alpha_2=jnp.array(0.6),
    K0_2=jnp.array(5.0),
    Ef_2=jnp.array(0.2),
    dB=jnp.array(1.0),
    dC=jnp.array(1.0),
    dD=jnp.array(1.0),
    K_het=jnp.array(20.0),
)

fig, axs = plt.subplots(2, 3, figsize=(10, 6), sharex=True, sharey=True)

# alpha_2
alpha_range = jnp.array([0.3, 0.4, 0.6, 0.7])
alpha_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(alpha_range, base_params.alpha_1),
    K0_1=jnp.full_like(alpha_range, base_params.K0_1),
    Ef_1=jnp.full_like(alpha_range, base_params.Ef_1),
    alpha_2=alpha_range,
    K0_2=jnp.full_like(alpha_range, base_params.K0_1),
    Ef_2=jnp.full_like(alpha_range, base_params.Ef_1),
    dB=jnp.full_like(alpha_range, base_params.dB),
    dC=jnp.full_like(alpha_range, base_params.dC),
    dD=jnp.full_like(alpha_range, base_params.dD),
    K_het=jnp.full_like(alpha_range, base_params.K_het),
)

alpha_currents = vmap(fd_solver.solve)(alpha_params)

for val, current in zip(alpha_range, alpha_currents):
    axs[0, 0].plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

axs[0, 0].legend()
axs[0, 0].set_title(r"$\alpha^{(2)}$")
axs[0, 0].set_ylabel(r"$J$")

# K0_2
K0_range = jnp.array([1.0, 5.0, 20.0, 50.0])
alpha_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(K0_range, base_params.alpha_1),
    K0_1=jnp.full_like(K0_range, base_params.K0_1),
    Ef_1=jnp.full_like(K0_range, base_params.Ef_1),
    alpha_2=jnp.full_like(K0_range, base_params.alpha_2),
    K0_2=K0_range,
    Ef_2=jnp.full_like(K0_range, base_params.Ef_1),
    dB=jnp.full_like(K0_range, base_params.dB),
    dC=jnp.full_like(K0_range, base_params.dC),
    dD=jnp.full_like(K0_range, base_params.dD),
    K_het=jnp.full_like(K0_range, base_params.K_het),
)

K0_currents = vmap(fd_solver.solve)(alpha_params)

for val, current in zip(K0_range, K0_currents):
    axs[0, 1].plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

axs[0, 1].legend()
axs[0, 1].set_title(r"$K_0^{(2)}$")
# dC
dC_range = jnp.array([0.1, 0.3, 0.6, 0.8])

dC_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(dC_range, base_params.alpha_1),
    K0_1=jnp.full_like(dC_range, base_params.K0_1),
    Ef_1=jnp.full_like(dC_range, base_params.Ef_1),
    alpha_2=jnp.full_like(dC_range, base_params.alpha_2),
    K0_2=jnp.full_like(dC_range, base_params.K0_2),
    Ef_2=jnp.full_like(dC_range, base_params.Ef_1),
    dB=jnp.full_like(dC_range, base_params.dB),
    dC=dC_range,
    dD=jnp.full_like(dC_range, base_params.dD),
    K_het=jnp.full_like(dC_range, base_params.K_het),
)

dC_currents = vmap(fd_solver.solve)(dC_params)

for val, current in zip(dC_range, dC_currents):
    axs[0, 2].plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

axs[0, 2].set_title(r"$d_C$")
axs[0, 2].legend()
axs[0, 2].set_xlabel(r"$\theta$")
# dD
dD_range = jnp.array([0.1, 0.3, 0.6, 0.8])

dD_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(dD_range, base_params.alpha_1),
    K0_1=jnp.full_like(dD_range, base_params.K0_1),
    Ef_1=jnp.full_like(dD_range, base_params.Ef_1),
    alpha_2=jnp.full_like(dD_range, base_params.alpha_2),
    K0_2=jnp.full_like(dD_range, base_params.K0_2),
    Ef_2=jnp.full_like(dD_range, base_params.Ef_1),
    dB=jnp.full_like(dD_range, base_params.dB),
    dC=jnp.full_like(dD_range, base_params.dC),
    dD=dD_range,
    K_het=jnp.full_like(dD_range, base_params.K_het),
)

dD_currents = vmap(fd_solver.solve)(dD_params)

for val, current in zip(dD_range, dD_currents):
    axs[1, 0].plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

axs[1, 0].set_title(r"$d_D$")
axs[1, 0].legend()
axs[1, 0].set_xlabel(r"$\theta$")
# Khet
Khet_range = jnp.array([1.0, 5.0, 20.0, 50.0])
Khet_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(Khet_range, base_params.alpha_1),
    K0_1=jnp.full_like(Khet_range, base_params.K0_1),
    Ef_1=jnp.full_like(Khet_range, base_params.Ef_1),
    alpha_2=jnp.full_like(Khet_range, base_params.alpha_2),
    K0_2=jnp.full_like(Khet_range, base_params.K0_2),
    Ef_2=jnp.full_like(Khet_range, base_params.Ef_1),
    dB=jnp.full_like(Khet_range, base_params.dB),
    dC=jnp.full_like(Khet_range, base_params.dC),
    dD=jnp.full_like(Khet_range, base_params.dD),
    K_het=Khet_range,
)

Khet_currents = vmap(fd_solver.solve)(Khet_params)

for val, current in zip(Khet_range, Khet_currents):
    axs[1, 1].plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

axs[1, 1].legend()
axs[1, 1].set_title(r"$K_{het}$")
axs[1, 1].set_xlabel(r"$\theta$")
axs[1, 1].set_ylabel(r"$J$")

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Figure 6: Heterogeneous Biplot


In [ ]:
hmc = np.load("./data/H_HMC_0.10_10.npz")

fig, axs = plt.subplots(2, 5, figsize=(12, 5))
options = {"bins": 50}

axs[0, 0].hist(hmc["alpha_1"].flatten(), **options)
axs[1, 0].hist(hmc["alpha_2"].flatten(), **options)
axs[0, 1].hist(hmc["K0_1"].flatten(), **options)
axs[1, 1].hist(hmc["K0_2"].flatten(), **options)
axs[0, 2].hist(hmc["K0_1"].flatten(), **options)
axs[1, 2].hist(hmc["K0_2"].flatten(), **options)
plt.tight_layout()
plt.show()

## Figure 10: Adsorption Example Voltammagrams

In [ ]:
voltammetry = CyclicDC(sigma=10, theta_i=20, theta_v=-20)
fdm_solver = AdsorptionReactionNewtonFDSolver(voltammetry, h0=1e-6, dtheta=1e-1)

fig, axs = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.5),
    K0_sol=jnp.array(0.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.5),
    K0_ads=jnp.array(1e6),
    K_A_ads=jnp.array(1e3),
    K_A_des=jnp.array(1e-3),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1e-3),
    dB=jnp.array(0.8),
)

sol, current = fdm_solver.solve(params)

axs[0, 0].plot(fdm_solver.applied_potentials, current)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-3),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(4.5),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
    dB=jnp.array(1.0),
)

sol, current = fdm_solver.solve(params)

axs[0, 1].plot(fdm_solver.applied_potentials, current)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.9),
    K0_sol=jnp.array(1000.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.9),
    K0_ads=jnp.array(0.0),
    K_A_ads=jnp.array(1.0),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
    dB=jnp.array(1.0),
)

sol, current = fdm_solver.solve(params)

axs[1, 0].plot(fdm_solver.applied_potentials, current)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-2),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(1.0),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(5.0),
    K_B_des=jnp.array(1e-1),
    dB=jnp.array(1.0),
)

sol, current = fdm_solver.solve(params)

axs[1, 1].plot(fdm_solver.applied_potentials, current)

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


## Figure N: Numerics Comparison for Adsorption

In [ ]:
voltammetry = CyclicDC(sigma=10, theta_i=20, theta_v=-20)
fd_newton = AdsorptionReactionNewtonFDSolver(voltammetry, h0=1e-6, dtheta=1e-1)
fd_explicit = AdsorptionReactionExplicitFDSolver(voltammetry, h0=1e-6, dtheta=1e-1)
fd_backward_implicit = AdsorptionReactionBackwardImplicitFDSolver(
    voltammetry, h0=1e-6, dtheta=1e-1
)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-3),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(4.5),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
    dB=jnp.array(1.0),
)

newton_sol, newton_current = fd_newton.solve(params)
back_sol, backward_current = fd_backward_implicit.solve(params)
exp_sol, explicit_current = fd_explicit.solve(params)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

# Current Plots
args = {"alpha": 0.5}
ax1.plot(
    fd_backward_implicit.applied_potentials,
    backward_current,
    label="Implicit",
    **args,
)
ax1.plot(
    fd_explicit.applied_potentials,
    explicit_current,
    label="Explicit",
    **args,
)
ax1.plot(
    fd_newton.applied_potentials, newton_current, label="Newton", **args, linestyle="--"
)

ax1.set_title("Voltammogram")
ax1.set_ylabel(r"$J$")
ax1.set_xlabel(r"$\theta$")
ax1.xaxis.set_inverted(True)
ax1.yaxis.set_inverted(True)


# Absolute Difference
backward_diff = jnp.abs((backward_current - newton_current) / newton_current)
explicit_diff = jnp.abs((explicit_current - newton_current) / newton_current)
ax2.plot(backward_diff, label="Implicit")
ax2.plot(explicit_diff, label="Explicit")

ax2.set_title("Current")
ax2.set_xlabel("$T$")
ax2.set_ylabel("Difference")
ax2.set_yscale("log")


# Non-linear term approximation erorr
newton_nonlinear = newton_sol[1:-1, 2] * newton_sol[1:-1, 0]

backward_diff = jnp.abs(
    (
        back_sol[1:-1, 2] * back_sol[:-2, 0]
        + back_sol[:-2, 2] * back_sol[1:-1, 0]
        - back_sol[:-2, 2] * back_sol[:-2, 0]
        - newton_nonlinear
    )
    / newton_nonlinear
)
exp_diff = jnp.abs(
    (exp_sol[:-2, 2] * exp_sol[:-2, 0] - newton_nonlinear) / newton_nonlinear
)

ax3.plot(backward_diff, label="Implicit")
ax3.plot(exp_diff, label="Explicit")

ax3.set_title("Linear Approximation")
ax3.set_xlabel("$T$")
ax3.set_ylabel("Difference")
ax3.set_yscale("log")

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
def current_sim(params, fd):
    sol, current = fd.solve(params)
    current.block_until_ready()
    return current

In [ ]:
%%timeit
current_sim(params, fd_explicit)

In [ ]:
%%timeit
current_sim(params, fd_backward_implicit)